In [0]:
# Instala a biblioteca de cliente para APIs de LLM (necessária para o código abaixo).
# %pip é o método oficial em notebooks Databricks — nunca use !pip.
# Precisa ser executada ANTES da célula que faz o import.
%pip install openai

In [0]:

from openai import OpenAI
import os

# AUTENTICAÇÃO (TOKEN)

# Para chamar um modelo, precisamos provar quem somos.
# O token é a 'chave' que autoriza a requisição.
#
# Opção A: ler o token de uma variável de ambiente
# (padrão quando o código roda FORA do Databricks,
# ex.: no seu laptop ou num app externo).
DATABRICKS_TOKEN = os.environ.get('DATABRICKS_TOKEN')

# Opção B: rodando DENTRO de um notebook Databricks, o token
# já está disponível no contexto — esta linha pega ele direto:
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
#
# Dica de segurança: NUNCA escreva o token literalmente no código
# (ex.: DATABRICKS_TOKEN = "dapii123..."). Sempre busque do ambiente.

# ============================================================
# 3. CRIANDO O CLIENTE (a 'conexão' com o LLM)
# ============================================================
# Truque principal: usamos a API da OpenAI, mas com
# base_url apontando para o AI GATEWAY do Databricks.
#
# Ou seja: as requisições vão para o seu workspace,
# que centraliza os modelos, governança, limites e logs.
#
# Rota: /ai-gateway/mlflow/v1 -> API 'Chat Completions' padrão.
client = OpenAI(
    api_key=DATABRICKS_TOKEN,               # o token do Databricks (NÃO é chave OpenAI)
    base_url="https://dbc-c4769987-718a.cloud.databricks.com/ai-gateway/mlflow/v1"
)

# ============================================================
# 4. A REQUISIÇÃO DE CHAT
# ============================================================
# Um 'chat' é uma lista de mensagens, cada uma com um PAPEL (role):
#
#   "system" -> instrução global: define o comportamento/persona
#                do modelo. É o 'manual de conduta' da conversa.
#   "user"   -> a pergunta do humano.
#
# (Existe também o "assistant": respostas anteriores do modelo,
# úteis para dar memória/contexto em conversas longas.)
response = client.chat.completions.create(

    # Nome do modelo em 3 partes: catálogo.esquema.serviço
    # 'system.ai' = modelos prontos, hospedados e governados
    # pelo Databricks (você não precisa de conta externa).
    model="system.ai.gpt-oss-120b",

    messages=[
        {
            "role": "system",
            "content": "use funny language "   # instrução de estilo: responda com humor
        },
        {
            "role": "user",
            "content": "why ai agents getting so popular ? "  # a pergunta em si
        }
    ],

    # Limite de tamanho da resposta (tokens ~ pedaços de palavras).
    # Controla custo e evita respostas gigantes.
    max_tokens=5000
)

# ============================================================
# 5. LENDO A RESPOSTA
# ============================================================
# A resposta chega como JSON estruturado, contendo metadados
# (modelo usado, contagem de tokens, etc.) e as 'choices'.
#
# choices[0]                -> primeira (e usualmente única) alternativa de resposta
#   .message.content        -> o texto gerado pelo modelo
#
# É por isso que precisamos 'navegar' até o conteúdo:
print(response.choices[0].message.content)

In [0]:
print(response.choices[0].message.content)

# Por que agentes de IA estão ficando tão populares?

AI agents are gaining popularity because they:

- **Automate complex tasks** — handle scheduling, data analysis, and customer support without constant human oversight.
- **Boost productivity** — integrate with tools (email, calendars, APIs) to streamline workflows.
- **Offer personalization** — adapt responses to user preferences and context.
- **Scale knowledge** — leverage large language models to provide up-to-date information across domains.
- **Lower entry barriers** — user-friendly interfaces let non-technical people deploy powerful automation.
- **Drive innovation** — enable new services (e.g., autonomous agents, digital assistants) that were previously impractical.


In [0]:
%sql
select ai_query('databricks-gpt-oss-120b', 'Por que os agentes de IA estão tão populares')

Os agentes de IA (assistentes virtuais, chatbots, “agents” que executam tarefas autonomamente) ganharam enorme popularidade nos últimos anos por uma combinação de fatores tecnológicos, econômicos e sociais. Veja os principais motivos:

| Motivo | Como contribui para a popularidade |
|--------|------------------------------------|
| **Avanços em aprendizado de máquina** | Modelos de linguagem como GPT‑4, LLaMA, Gemini, etc., conseguem gerar texto coerente, entender contexto e adaptar respostas, tornando a interação mais natural. |
| **Disponibilidade de grandes volumes de dados** | Dados de texto, voz e imagens em escala permitem treinar modelos mais precisos e especializados. |
| **Infraestrutura em nuvem** | Serviços como AWS, Azure, Google Cloud oferecem GPUs/TPUs sob demanda, reduzindo o custo e a complexidade de rodar modelos pesados. |
| **Facilidade de integração** | APIs padronizadas (OpenAI, Anthropic, Azure OpenAI) permitem que desenvolvedores adicionem IA a aplicativos, sites e dispositivos com poucas linhas de código. |
| **Produtividade e automação** | Agentes podem automatizar tarefas repetitivas (suporte ao cliente, agendamento, análise de documentos), liberando tempo humano para atividades de maior valor. |
| **Personalização em escala** | Eles aprendem preferências individuais e entregam respostas ou recomendações adaptadas, melhorando a experiência do usuário. |
| **Redução de custos operacionais** | Substituem ou complementam equipes de atendimento, reduzindo a necessidade de contratação massiva de operadores. |
| **Hype e mídia** | Cobertura massiva na imprensa, demonstrações virais (ChatGPT, Copilot, DALL·E) criam um efeito de “efeito manada”, incentivando empresas a adotarem rapidamente para não ficarem para trás. |
| **Acessibilidade para não‑técnicos** | Ferramentas low‑code/no‑code (ChatGPT Playground, Microsoft Power Automate, Zapier) permitem que profissionais de áreas diversas criem fluxos de trabalho com IA sem saber programar. |
| **Regulamentação e confiança** | Cada vez mais normas de privacidade (GDPR, LGPD) e frameworks de IA responsável dão segurança para empresas implementarem soluções de forma ética. |
| **Ecossistema de plugins e extensões** | Plataformas como ChatGPT Plugins, Microsoft Teams Apps, Slack Bots criam um mercado de funcionalidades adicionais, ampliando o valor dos agentes. |

### Tendências que reforçam essa popularidade
1. **IA generativa multimodal** – Capacidade de lidar com texto, imagem, áudio e até vídeo em um único agente.
2. **IA “retrieval‑augmented”** – Integração com bases de conhecimento corporativas, permitindo respostas baseadas em dados internos e não apenas no treinamento geral.
3. **Edge AI** – Execução de modelos menores diretamente em dispositivos (smartphones, wearables), reduzindo latência e dependência de conexão.
4. **Modelos de “instruction‑following”** – Agentes que entendem instruções em linguagem natural, facilitando a criação de fluxos de trabalho complexos sem programação.

### Resumindo
Os agentes de IA são populares porque a tecnologia chegou a um ponto em que **é suficientemente boa, acessível e econômica** para resolver problemas reais de negócios e do dia‑a‑dia, ao mesmo tempo em que gera entusiasmo cultural e mediático. Essa convergência cria um ciclo virtuoso: mais adoção gera mais investimento, que por sua vez gera modelos ainda melhores e mais casos de uso.

Se quiser saber como aplicar um agente de IA a um caso específico (por exemplo, suporte ao cliente, automação de relatórios ou assistente pessoal), basta me dizer!